# Assignment — Do Convergent Sites Cluster in Protein Domains?

In Notebook 3 you identified alignment positions where echolocating
species show convergent amino acid usage. Now the biological question:

> **Are these convergent sites randomly scattered across the protein,
> or do they cluster in specific functional domains?**

If convergence is driven by functional pressure (e.g., optimizing the
motor function of prestin), we'd expect convergent sites to be
**enriched** in the transmembrane motor domain rather than spread
evenly.

## Your task

Design and implement a **permutation test** to answer this question.
You need to:
1. Map convergent sites to protein domains
2. Choose an appropriate test statistic
3. Build a null distribution by shuffling
4. Compute a p-value and visualize the result

**Deliverables:**
- The completed code in this notebook
- A plot showing your null distribution and observed statistic
- A short written interpretation (2-3 sentences in a markdown cell)

---
## 1. Setup and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import math

sns.set_theme()
np.random.seed(42)

DATA = os.path.join('..', 'data')
SUB_DIR = os.path.join(DATA, 'subfamilies')

def read_fasta(path):
    seqs = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:].split()[0]
                seqs[header] = ''
            elif header:
                seqs[header] += line
    return seqs

### Load convergence results from Notebook 3

In [ ]:
results_df = pd.read_csv(os.path.join(DATA, 'prestin_convergence_results.csv'))
print(f"Loaded {len(results_df)} position results")
print(f"Columns: {list(results_df.columns)}")

### Define significant convergent sites

Use a p-value threshold to define which positions are "convergent."

In [ ]:
threshold = 0.05
convergent_positions = results_df[results_df['pvalue'] < threshold]['position'].values
n_convergent = len(convergent_positions)
n_total = len(results_df)

print(f"Convergent positions (p < {threshold}): {n_convergent} / {n_total}")

---
## 2. Protein domain boundaries

Human prestin (SLC26A5) has the following approximate domain structure:

| Domain | Alignment positions* | Function |
|:---|:---|:---|
| N-terminal | 0 – 79 | Cytoplasmic, regulatory |
| TMD (transmembrane) | 80 – 504 | Motor domain, 14 TM helices |
| Linker | 505 – 529 | Connects TMD to STAS |
| STAS domain | 530 – end | C-terminal regulatory |

*These are **approximate** boundaries in the alignment coordinate system.
In practice you would map alignment positions to the human reference
sequence using a gap-mapping function. For this assignment, use the
approximate boundaries below.

In [ ]:
# Domain definitions (approximate alignment coordinates)
# Adjust these based on your actual alignment if needed!
DOMAINS = {
    'N-terminal': (0, 79),
    'TMD':        (80, 504),
    'Linker':     (505, 529),
    'STAS':       (530, n_total - 1),
}

# Which domain does each position belong to?
def get_domain(pos):
    for name, (start, end) in DOMAINS.items():
        if start <= pos <= end:
            return name
    return 'other'

---
## 3. Your analysis

### Step A: Count convergent sites per domain

How many convergent sites fall in each domain?

In [ ]:
# YOUR CODE HERE
# Count how many convergent sites fall in each domain
# Compare with how many total positions each domain has

### Step B: Choose a test statistic

You need a single number that captures whether convergent sites are
**more concentrated** in one domain than expected. Some options to
consider:

- The **number** of convergent sites in the TMD
- The **fraction** of convergent sites in the TMD vs. overall
- A **chi-squared-like** measure across all domains
- Something else?

Think about what makes biological sense. Write your choice and
reasoning here.

In [ ]:
# YOUR CODE: define your test statistic as a function
# It should take a list of positions and return a single number

### Step C: Build the null distribution

Shuffle the positions of convergent sites — if convergence had nothing
to do with domain structure, the sites could be anywhere in the
alignment. Recompute your statistic for each shuffled set.

**Important:** Think carefully about *what* to shuffle. You have
N convergent sites out of M total positions. The null hypothesis is
that these N positions are a random sample from all M positions.

In [ ]:
# YOUR CODE: permutation test
# 1. Compute observed statistic
# 2. Shuffle: randomly pick n_convergent positions from all positions
# 3. Recompute statistic
# 4. Repeat 10,000 times
# 5. Compute p-value

### Step D: Visualize

Plot the null distribution as a histogram, mark the observed value,
and show the p-value.

In [ ]:
# YOUR CODE: create the plot

### Step E: Interpretation

Write 2-3 sentences interpreting your result.
- Is there significant clustering of convergent sites?
- In which domain, if any?
- What does this suggest about the functional role of convergent
  evolution in prestin?

*Your interpretation here*

---
## Bonus (optional)

If you finish early:

1. Repeat the analysis using a stricter threshold (p < 0.01) for
   defining convergent sites. Does the conclusion change?

2. Instead of testing just the TMD, test each domain separately.
   Apply Bonferroni correction for multiple domain tests.

3. The domain boundaries above are approximate. Load the prestin
   alignment and the human reference sequence, and write code to
   map alignment positions to the ungapped human sequence positions.
   Use those for more accurate domain mapping.